In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc numpy
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     channel="ibm_quantum_platform",
#     token="<your-api-key>",
#     # instance="<IBM Cloud CRN or instance name>",  # optional
#     set_as_default=True,
#     overwrite=True,
# )

# Buat pass manager untuk dynamical decoupling

<Accordion>
<AccordionItem title="Versi paket">

Kode di halaman ini dikembangkan menggunakan persyaratan berikut.
Kami merekomendasikan menggunakan versi ini atau yang lebih baru.

```
qiskit[all]~=2.4.0
qiskit-ibm-runtime~=0.46.1
```
</AccordionItem>
</Accordion>
Halaman ini menunjukkan cara menggunakan pass [`PadDynamicalDecoupling`](https://docs.quantum.ibm.com/api/qiskit/qiskit.transpiler.passes.PadDynamicalDecoupling) untuk menambahkan teknik penekanan error yang disebut _dynamical decoupling_ ke Circuit.

Dynamical decoupling bekerja dengan menambahkan rangkaian pulsa (dikenal sebagai _urutan dynamical decoupling_) ke Qubit yang menganggur untuk membaliknya di sekitar Bloch sphere, yang membatalkan efek noise channel sehingga menekan dekoherensi. Rangkaian pulsa ini mirip dengan refocusing pulse yang digunakan dalam resonansi magnetik nuklir. Untuk deskripsi lengkapnya, lihat [A Quantum Engineer's Guide to Superconducting Qubits](https://arxiv.org/abs/1904.06560).

Karena pass `PadDynamicalDecoupling` hanya beroperasi pada Circuit yang sudah dijadwalkan dan melibatkan Gate yang belum tentu merupakan basis Gate dari target kita, kamu juga memerlukan pass [`ALAPScheduleAnalysis`](https://docs.quantum.ibm.com/api/qiskit/qiskit.transpiler.passes.ALAPScheduleAnalysis) dan [`BasisTranslator`](https://docs.quantum.ibm.com/api/qiskit/qiskit.transpiler.passes.BasisTranslator).

Ambil informasi `target` dari `backend` dan simpan nama operasi sebagai `basis_gates` karena `target` perlu dimodifikasi untuk menambahkan informasi timing bagi Gate yang digunakan dalam dynamical decoupling.

> **Note:** Mock backend `FakeSherbrooke` dari `qiskit_ibm_runtime` digunakan dalam contoh-contoh ini, tetapi kamu bisa mencobanya pada Backend nyata atau palsu apa pun yang kompatibel dengan Qiskit. Hasilmu mungkin berbeda.

In [1]:
from qiskit_ibm_runtime.fake_provider import FakeSherbrooke

backend = FakeSherbrooke()

target = backend.target
basis_gates = list(target.operation_names)

Buat Circuit `efficient_su2` sebagai contoh. Pertama, transpile Circuit ke Backend karena pulsa dynamical decoupling perlu ditambahkan setelah Circuit ditranspile dan dijadwalkan. Dynamical decoupling sering bekerja paling baik ketika ada banyak waktu idle dalam Circuit kuantum — artinya, ada Qubit yang tidak digunakan sementara yang lain aktif. Inilah yang terjadi dalam Circuit ini karena Gate `ecr` dua-Qubit diterapkan secara berurutan dalam ansatz ini.

In [2]:
from qiskit.transpiler import generate_preset_pass_manager
from qiskit.circuit.library import efficient_su2

qc = efficient_su2(12, entanglement="circular", reps=1)
pm = generate_preset_pass_manager(1, target=target, seed_transpiler=12345)
qc_t = pm.run(qc)
qc_t.draw("mpl", fold=-1, idle_wires=False)

<Image src="../docs/images/guides/dynamical-decoupling-pass-manager/extracted-outputs/9479a60c-d5d0-45c7-a93e-a2a488ba8985-0.svg" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/guides/dynamical-decoupling-pass-manager/extracted-outputs/9479a60c-d5d0-45c7-a93e-a2a488ba8985-0.svg)

Urutan dynamical decoupling adalah serangkaian Gate yang bersama-sama membentuk identitas dan ditempatkan secara teratur dalam waktu. Sebagai contoh, mulailah dengan membuat urutan sederhana yang disebut XY4 yang terdiri dari empat Gate.

In [3]:
from qiskit.circuit.library import XGate, YGate

X = XGate()
Y = YGate()

dd_sequence = [X, Y, X, Y]

Karena pengaturan waktu urutan dynamical decoupling yang teratur, informasi tentang `YGate` harus ditambahkan ke `target` karena Gate ini *bukan* basis Gate, sedangkan `XGate` adalah. Namun kita sudah tahu *a priori* bahwa `YGate` memiliki durasi dan error yang sama dengan `XGate`, sehingga kita bisa mengambil properti tersebut dari `target` dan menambahkannya kembali untuk objek `YGate`. Inilah juga alasan `basis_gates` disimpan secara terpisah, karena kita menambahkan instruksi `YGate` ke `target` meskipun itu bukan basis Gate sebenarnya dari `ibm_fez`.

In [4]:
from qiskit.transpiler import InstructionProperties

y_gate_properties = {}
for qubit in range(target.num_qubits):
    y_gate_properties.update(
        {
            (qubit,): InstructionProperties(
                duration=target["x"][(qubit,)].duration,
                error=target["x"][(qubit,)].error,
            )
        }
    )

target.add_instruction(YGate(), y_gate_properties)

Circuit ansatz seperti `efficient_su2` bersifat terparameterisasi, sehingga harus diikat dengan nilai sebelum dikirim ke Backend. Di sini, tetapkan parameter acak.

In [5]:
import numpy as np

rng = np.random.default_rng(1234)
qc_t.assign_parameters(
    rng.uniform(-np.pi, np.pi, qc_t.num_parameters), inplace=True
)

Selanjutnya, jalankan pass kustom. Buat instance `PassManager` dengan `ALAPScheduleAnalysis` dan `PadDynamicalDecoupling`. Jalankan `ALAPScheduleAnalysis` terlebih dahulu untuk menambahkan informasi timing tentang Circuit kuantum sebelum urutan dynamical decoupling yang berjarak teratur dapat ditambahkan. Pass-pass ini dijalankan pada Circuit menggunakan `.run()`.

In [6]:
from qiskit.transpiler import PassManager
from qiskit.transpiler.passes.scheduling import (
    ALAPScheduleAnalysis,
    PadDynamicalDecoupling,
)

dd_pm = PassManager(
    [
        ALAPScheduleAnalysis(target=target),
        PadDynamicalDecoupling(target=target, dd_sequence=dd_sequence),
    ]
)
qc_dd = dd_pm.run(qc_t)

Gunakan alat visualisasi [`timeline_drawer`](https://docs.quantum.ibm.com/api/qiskit/qiskit.visualization.timeline_drawer) untuk melihat timing Circuit dan memastikan bahwa urutan objek `XGate` dan `YGate` yang berjarak teratur muncul dalam Circuit.

In [7]:
from qiskit.visualization import timeline_drawer

timeline_drawer(qc_dd, idle_wires=False, target=target)

<Image src="../docs/images/guides/dynamical-decoupling-pass-manager/extracted-outputs/7a552621-a96f-4bb8-ae9b-4ab5a65bbb64-0.svg" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/guides/dynamical-decoupling-pass-manager/extracted-outputs/7a552621-a96f-4bb8-ae9b-4ab5a65bbb64-0.svg)

Terakhir, karena `YGate` bukan basis Gate sebenarnya dari Backend kita, terapkan pass `BasisTranslator` secara manual (ini adalah pass default, tetapi dieksekusi sebelum penjadwalan, sehingga perlu diterapkan lagi). Session equivalence library adalah library ekuivalensi Circuit yang memungkinkan Transpiler menguraikan Circuit menjadi basis Gate, sebagaimana juga ditentukan sebagai argumen.

In [8]:
from qiskit.circuit.equivalence_library import (
    SessionEquivalenceLibrary as sel,
)
from qiskit.transpiler.passes import BasisTranslator

qc_dd = BasisTranslator(sel, basis_gates)(qc_dd)
qc_dd.draw("mpl", fold=-1, idle_wires=False)

<Image src="../docs/images/guides/dynamical-decoupling-pass-manager/extracted-outputs/f0f4b29d-1c95-47d2-a7ad-8e130eaff74a-0.svg" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/guides/dynamical-decoupling-pass-manager/extracted-outputs/f0f4b29d-1c95-47d2-a7ad-8e130eaff74a-0.svg)

Sekarang, objek `YGate` tidak ada lagi dalam Circuit kita, dan ada informasi timing eksplisit dalam bentuk Gate `Delay`. Circuit yang sudah ditranspile dengan dynamical decoupling ini kini siap dikirim ke Backend.

## Langkah berikutnya

> **Tip:** - Untuk mempelajari cara menggunakan fungsi `generate_preset_passmanager` alih-alih menulis pass sendiri, mulailah dengan topik [Pengaturan default dan opsi konfigurasi Transpiler](defaults-and-configuration-options).
> - Coba panduan [Bandingkan pengaturan Transpiler](/guides/circuit-transpilation-settings#compare-transpiler-settings).
> - Lihat [Dokumentasi API Transpile](https://docs.quantum-computing.ibm.com/api/qiskit/transpiler).